In [1]:
from sklearn.datasets import fetch_california_housing, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

def load_and_split(X, y, random_state=42):
    print(X.shape, y.shape)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def fetch_openml_numeric(name, target):
    ds = fetch_openml(name, as_frame=True)
    X = ds.data
    y = ds.target.astype(float)

    # оставляем только числовые признаки
    X = X.select_dtypes(include=[np.number])

    return X.values, y.values


def fetch_openml_numeric_by_id(data_id):
    ds = fetch_openml(data_id=data_id, as_frame=True)
    X = ds.data.select_dtypes(include=[np.number])
    y = ds.target.astype(float)
    return X.values, y.values

In [2]:
# "catboost-style (self-made, default catboost HP)",california_housing,0.842888,0.848189,849
# catboost (default HP),california_housing,0.842664,0.850693,872
# "catboost-style (self-made, default catboost HP)",bike_sharing,0.932619,0.924521,922
# catboost (default HP),bike_sharing,0.932571,0.924632,830
# "catboost-style (self-made, default catboost HP)",medical_charges,0.966301,0.975550,275
# catboost (default HP),medical_charges,0.966301,0.975550,275
# "catboost-style (self-made, default catboost HP)",king_county_house_prices,0.891442,0.858218,162
# catboost (default HP),king_county_house_prices,0.896683,0.871238,216


In [ ]:
RESULTS_PATH = "benchmark_results.csv"
N_ESTIMATORS = 1000
EARLY_STOPPING_ROUNDS = 20
OPTUNA_TRIALS = 50
LEARNING_RATE = 0.1

import os
import sys
sys.path.append("..")


import csv
from src.gradient_boosting_regressor import MyCatBoost

from catboost import CatBoostRegressor

def log_result_csv(model_name, name, r2_val, r2_test, n_trees, train_time, path=RESULTS_PATH):
    file_exists = os.path.isfile(path)

    with open(path, mode="a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["model", "dataset", "r2_val", "r2_test", "n_trees", "train_time (sec)"])
        writer.writerow([model_name, name, f"{r2_val:.6f}", f"{r2_test:.6f}", n_trees, train_time])


from time import time

# Experiment 1: Default Catboost
def test_base_catboost(name, value):
    time_1 = time()
    X_train, X_val, X_test, y_train, y_val, y_test = value

    gbrt = CatBoostRegressor(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        loss_function="RMSE",
        verbose=False
    )

    gbrt.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )

    pred = gbrt.predict(X_test)
    r2_val = r2_score(y_val, gbrt.predict(X_val))
    r2_test = r2_score(y_test, pred)
    number_of_trees = gbrt.get_best_iteration() + 1

    time_2 = time()
    
    log_result_csv(
        "Exp 1:  catboost (default HP)", name, r2_val, r2_test, number_of_trees, time_2 - time_1
    )

def test_my_catboost(name, value):
    time_1 = time()
    X_train, X_val, X_test, y_train, y_val, y_test = value

    def base_model_fn(noise_std):
        return CatBoostRegressor(
            iterations=1, 
            learning_rate=1.0,
            random_strength=noise_std,
            verbose=False
        )
    
    gbrt = MyCatBoost(
        base_model_fn=base_model_fn,
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        random_strength=1,
        verbose=False
    )
    gbrt.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )

    pred = gbrt.predict(X_test)
    r2_val = r2_score(y_val, gbrt.predict(X_val))
    r2_test = r2_score(y_test, pred)
    number_of_trees = len(gbrt.models)

    time_2 = time()

    log_result_csv(
        "Exp 2:  catboost-style (self-made, default catboost HP)", 
        name, r2_val, r2_test, number_of_trees, time_2 - time_1
    )

import optuna

def test_my_catboost_random_hp(name, value):
    time_1 = time()
    X_train, X_val, X_test, y_train, y_val, y_test = value
    def base_model_fn(noise_std):
        # Создаём study с RandomSampler для каждого дерева
        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.RandomSampler()
        )
        
        # Берём один trial (случайная выборка из search space)
        trial = study.ask()
        
        grow_policy = trial.suggest_categorical(
            "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
        )

        params = {
            "iterations": 1,
            "learning_rate": 1.0,
            "loss_function": "RMSE",
            "boost_from_average": False,
            "boosting_type": "Plain",
            "grow_policy": grow_policy,
            "border_count": trial.suggest_int("border_count", 32, 255),
            "feature_border_type": trial.suggest_categorical(
                "feature_border_type", ["GreedyLogSum", "Median", "Uniform"]
            ),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-6, 100.0, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 64),
            "random_strength": trial.suggest_float(
                "random_strength", 1e-3, 10.0, log=True
            ) * noise_std,
            "rsm": trial.suggest_float("rsm", 0.3, 1.0),
            "score_function": trial.suggest_categorical(
                "score_function", ["Cosine", "L2"]
            ),
            "random_seed": int.from_bytes(os.urandom(4), "little"),
            "verbose": False,
        }

        if grow_policy in ["SymmetricTree", "Depthwise"]:
            params["depth"] = trial.suggest_int("depth", 2, 12)
        else:
            params["max_leaves"] = trial.suggest_int("max_leaves", 8, 64)

        bootstrap_type = trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli", "MVS", "No"]
        )
        params["bootstrap_type"] = bootstrap_type

        if bootstrap_type == "Bernoulli" or bootstrap_type == "MVS":
            params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)
        elif bootstrap_type == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float(
                "bagging_temperature", 0.0, 10.0
            )

        return CatBoostRegressor(**params)
    
    gbrt = MyCatBoost(
        base_model_fn=base_model_fn,
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        random_strength=1,
        bootstrap_type="No",
        verbose=False,
        fix_borders=False
    )
    gbrt.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )

    pred = gbrt.predict(X_test)
    r2_val = r2_score(y_val, gbrt.predict(X_val))
    r2_test = r2_score(y_test, pred)
    number_of_trees = len(gbrt.models)

    time_2 = time()

    log_result_csv(
        "Exp 3:  catboost-style (self-made, random HP)", 
        name, r2_val, r2_test, number_of_trees, time_2 - time_1
    )

# Exp 4: HPO Catboost
def test_base_catboost_HPO(name, value):
    X_train, X_val, X_test, y_train, y_val, y_test = value

    def objective(trial):
        params = {
            "n_estimators": N_ESTIMATORS,
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "loss_function": "RMSE",
            "boosting_type": "Plain",
            "grow_policy": trial.suggest_categorical(
                "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
            ),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "feature_border_type": trial.suggest_categorical(
                "feature_border_type", ["GreedyLogSum", "Median", "Uniform"]
            ),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-6, 100.0, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 64),
            "random_strength": trial.suggest_float(
                "random_strength", 1e-3, 10.0, log=True
            ),
            "rsm": trial.suggest_float("rsm", 0.3, 1.0),
            "score_function": trial.suggest_categorical(
                "score_function", ["Cosine", "L2"]
            ),
            "random_seed": int.from_bytes(os.urandom(4), "little"),  # 32-bit entropy,
            "verbose": False,
        }

        if params["grow_policy"] in ["SymmetricTree", "Depthwise"]:
            params["depth"] = trial.suggest_int("depth", 2, 12)
        else:
            params["max_leaves"] = trial.suggest_int("max_leaves", 8, 64)

        bootstrap_type = trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli", "No"]
        )
        params["bootstrap_type"] = bootstrap_type

        if bootstrap_type == "Bernoulli":
            params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)
        elif bootstrap_type == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float(
                "bagging_temperature", 0.0, 10.0
            )

        model = CatBoostRegressor(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        )

        pred = model.predict(X_val)
        r2 = r2_score(y_val, pred)
        return r2
    
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=OPTUNA_TRIALS)
    best_params = study.best_params

    gbrt = CatBoostRegressor(
        n_estimators=N_ESTIMATORS,
        loss_function="RMSE",
        **best_params,
        verbose=False
    )

    gbrt.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )

    pred = gbrt.predict(X_test)
    r2_val = r2_score(y_val, gbrt.predict(X_val))
    r2_test = r2_score(y_test, pred)
    number_of_trees = gbrt.get_best_iteration() + 1
    log_result_csv(
        "Exp 4:  catboost (HPO)", name, r2_val, r2_test, number_of_trees
    )

def test_all_4(name, value):

    X_train, X_val, X_test, y_train, y_val, y_test = value

    X_train = X_train
    y_train = y_train
    
    value = (X_train, X_val, X_test, y_train, y_val, y_test)

    test_base_catboost(name, value)
    #test_my_catboost(name, value)
    test_my_catboost_random_hp(name, value)
    #test_base_catboost_HPO(name, value)

In [6]:
# 1. California Housing (~20k)
cal = fetch_california_housing(as_frame=True)
test_all_4(
    "california_housing",
    load_and_split(cal.data.values, cal.target.values),
)


# 2. Bike Sharing Demand (~17k)
X, y = fetch_openml_numeric("Bike_Sharing_Demand", target="count")
test_all_4(
    "bike_sharing",
    load_and_split(X, y),
)


# 3. Medical Charges (~13k)
X, y = fetch_openml_numeric("medical_charges", target="charges")
test_all_4(
    "medical_charges",
    load_and_split(X, y),
)


# King County House Prices
X, y = fetch_openml_numeric_by_id(42165)
test_all_4(
    "king_county_house_prices",
    load_and_split(X, y),
)


# 1. Online News Popularity (shares)
# ~39k samples, noisy, heavy-tailed target
X, y = fetch_openml_numeric_by_id(42705)
test_all_4(
    "online_news_popularity",
    load_and_split(X, y),
)


# 2. YearPredictionMSD
# ~515k samples, но low-dim, можно сабсемплить
X, y = fetch_openml_numeric_by_id(44027)
test_all_4(
    "year_prediction_msd",
    load_and_split(
        X,
        y,  # безопасный сабсет
    ),
)


# 3. CPU Activity
# ~20k samples, классический UCI-style regression
X, y = fetch_openml_numeric_by_id(44963)
test_all_4(
    "cpu_activity",
    load_and_split(X, y),
)


# 4. Facebook Comment Volume
# ~50k samples
X, y = fetch_openml_numeric_by_id(4549)
test_all_4(
    "facebook_comment_volume",
    load_and_split(X, y),
)


# 5. Airline Delay (departure delay)
# ~54k samples после очистки
X, y = fetch_openml_numeric_by_id(1169)
mask = np.isfinite(y)
test_all_4(
    "airline_delay",
    load_and_split(X[mask], y[mask]),
)


# 6. Superconductivity
# ~21k samples, физика, сложные взаимодействия
X, y = fetch_openml_numeric_by_id(44964)
test_all_4(
    "superconductivity",
    load_and_split(X, y),
)


# 7. Diamonds (price)
# ~54k samples
X, y = fetch_openml_numeric_by_id(42225)
test_all_4(
    "diamonds",
    load_and_split(X, y),
)


# 8. House Prices (Ames, extended)
# ~29k samples
X, y = fetch_openml_numeric_by_id(42563)
test_all_4(
    "ames_housing_large",
    load_and_split(X, y),
)


# 9. Brazilian Houses
# ~10k samples
X, y = fetch_openml_numeric_by_id(45020)
test_all_4(
    "brazilian_houses",
    load_and_split(X, y),
)


# 10. Metro Interstate Traffic Volume
# ~48k samples, сильная сезонность
X, y = fetch_openml_numeric_by_id(42477)
test_all_4(
    "metro_traffic_volume",
    load_and_split(X, y),
)

(20640, 8) (20640,)
california_housing (13209, 8) (13209,)
(17379, 8) (17379,)
bike_sharing (11122, 8) (11122,)


/home/artem/.local/lib/python3.10/site-packages/sklearn/datasets/_openml.py:323: UserWarning: Multiple active versions of the dataset matching the name bike_sharing_demand exist. Versions may be fundamentally different, returning version 2. Available versions:
- version 2, status: active
  url: https://www.openml.org/search?type=data&id=42712
- version 3, status: active
  url: https://www.openml.org/search?type=data&id=42713

  warn(warning_msg)
/home/artem/.local/lib/python3.10/site-packages/sklearn/datasets/_openml.py:323: UserWarning: Multiple active versions of the dataset matching the name medical_charges exist. Versions may be fundamentally different, returning version 5. Available versions:
- version 5, status: active
  url: https://www.openml.org/search?type=data&id=42130
- version 6, status: active
  url: https://www.openml.org/search?type=data&id=42131

  warn(warning_msg)


(163065, 5) (163065,)
medical_charges (104361, 5) (104361,)
(1460, 37) (1460,)
king_county_house_prices (934, 37) (934,)
(400000, 100) (400000,)
online_news_popularity (256000, 100) (256000,)
(515345, 90) (515345,)
year_prediction_msd (329820, 90) (329820,)
(45730, 9) (45730,)
cpu_activity (29267, 9) (29267,)
(583250, 77) (583250,)
facebook_comment_volume (373280, 77) (373280,)
(539383, 3) (539383,)
airline_delay (345204, 3) (345204,)
(21263, 81) (21263,)
superconductivity (13608, 81) (13608,)
(53940, 6) (53940,)
diamonds (34521, 6) (34521,)
(1460, 36) (1460,)
ames_housing_large (934, 36) (934,)
(13272, 20) (13272,)
brazilian_houses (8493, 20) (8493,)
(30000, 23) (30000,)
metro_traffic_volume (19200, 23) (19200,)
